In [2]:
%load_ext autoreload
%autoreload 2
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from tqdm import tqdm
import time

import yaml
from src.sampling.main import stratified_spatial_kfold_dual

from src.utils import read_config
from src.raingauge.utils import load_raingauge_dataset, filter_uptime, get_station_coordinate_mappings
from benchmarks.models.idw import run_IDW_benchmark

from scipy.stats import pearsonr

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
# IDW: 

# ── IDW baseline on the Wagga gauge dataset ───────────────────────────────────
# Predicts each held-out test station from the rest of the gauge network via
# inverse-distance weighting. Uses the SAME folds (seed=123) and SAME metrics as
# the GNN runs, so the numbers drop straight into the gauge / +radar comparison.
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import precision_recall_fscore_support

from src.raingauge.utils import load_raingauge_dataset
from src.sampling.main import stratified_spatial_kfold_dual

# --- knobs ---
POWER      = 2       # IDW distance exponent
N_NEAREST  = 10      # nearest source gauges to use (falls back to all available)
THRESHOLD  = 0.5     # mm/h wet/dry threshold for precision/recall/F1
FOLD_COUNT = 5

# --- 1. load Wagga gauge data (same paths as config_wagga_gauge.yaml) ---
gauge_df, meta_df = load_raingauge_dataset(
    rainfall_file="database/australia/rainfall_2022_sydney75_long.csv",
    metadata_file="database/australia/station_metadata_sydney75.csv",
    start=2022, end=2022, uptime_threshold=0.9,
)
gauge_df.columns = gauge_df.columns.astype(str)                       # station_id cols
coords = {str(r.id): (r.latitude, r.longitude) for r in meta_df.itertuples()}

# --- 2. identical fold split ---
split_info = stratified_spatial_kfold_dual(
    meta_df, seed=123, plot=False, n_splits=FOLD_COUNT
)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def idw_predict(test_id, src_ids):
    """IDW time series for one test station from source gauges (NaN-aware)."""
    tlat, tlon = coords[test_id]
    d = np.array([haversine_km(tlat, tlon, *coords[s]) for s in src_ids])
    order = np.argsort(d)[:min(N_NEAREST, len(src_ids))]
    src = [src_ids[i] for i in order]
    w = 1.0 / np.maximum(d[order], 1e-6) ** POWER                     # (k,)
    S = gauge_df[src].to_numpy(dtype=float)                          # (T, k)
    mask = ~np.isnan(S)
    den = (mask * w[None, :]).sum(axis=1)
    num = (np.where(mask, S, 0.0) * w[None, :]).sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, np.nan)                  # (T,)

# --- 3. run IDW per fold, collect pooled + per-station predictions ---
per_fold_rows, all_p, all_t = [], [], []
for fold in range(FOLD_COUNT):
    src_ids  = [str(s) for s in split_info[fold]["statistical"]["train"]]   # all non-test gauges
    test_ids = [str(s) for s in split_info[fold]["ml"]["test"]]
    fp, ft, st_rmse, st_r = [], [], [], []
    for tid in test_ids:
        pred = idw_predict(tid, src_ids)
        targ = gauge_df[tid].to_numpy(dtype=float)
        m = (~np.isnan(pred)) & (~np.isnan(targ))
        if m.sum() < 2:
            continue
        p, t = pred[m], targ[m]
        fp.append(p); ft.append(t)
        st_rmse.append(np.sqrt(np.mean((p - t) ** 2)))
        st_r.append(pearsonr(p, t)[0] if p.std() > 0 and t.std() > 0 else np.nan)
    p = np.concatenate(fp); t = np.concatenate(ft)
    all_p.append(p); all_t.append(t)
    prec, rec, f1, _ = precision_recall_fscore_support(
        t >= THRESHOLD, p >= THRESHOLD, average="binary", zero_division=0)
    per_fold_rows.append({
        "fold": fold,
        "pearson_r":          pearsonr(p, t)[0],
        "rmse":               np.sqrt(np.mean((p - t) ** 2)),
        "mae":                np.mean(np.abs(p - t)),
        "station_mean_rmse":   float(np.mean(st_rmse)),
        "station_median_rmse": float(np.median(st_rmse)),
        "station_median_r":    float(np.nanmedian(st_r)),
        "precision": prec, "recall": rec, "f1": f1,
        "n_test_stations": len(st_rmse),
    })

idw_per_fold = pd.DataFrame(per_fold_rows)
idw_summary  = idw_per_fold.drop(columns=["fold"]).mean().to_frame("mean_over_folds").T
idw_per_fold.to_csv("idw_sydney_per_fold.csv", index=False)

print("── IDW per fold ──");  display(idw_per_fold.round(4))
print("── IDW mean over folds ──");  display(idw_summary.round(4))

Loading Australian raingauge data from database/australia/rainfall_2022_sydney75_long.csv
Pivoted dataframe shape (before uptime filter): (8760, 75)
Loading station metadata from database/australia/station_metadata_sydney75.csv
Filtering by uptime threshold = 0.9
Dataframe shape after filter: (8760, 75)
Stations retained: 75
SPATIAL K-FOLD STRATIFIED SAMPLING (Fixed)
Total stations: 75
Clusters: 8
Folds: 5
Random seed: 123
--------------------------------------------------------------------------------
Cluster 0: 10 stations → 5 test assignments, 5 always-train
Cluster 1: 8 stations → 5 test assignments, 3 always-train
Cluster 2: 6 stations → 5 test assignments, 1 always-train
Cluster 3: 16 stations → 5 test assignments, 11 always-train
Cluster 4: 11 stations → 5 test assignments, 6 always-train
Cluster 5: 8 stations → 5 test assignments, 3 always-train
Cluster 6: 6 stations → 5 test assignments, 1 always-train
Cluster 7: 10 stations → 5 test assignments, 5 always-train

Creating Fold 

,fold,pearson_r,rmse,mae,station_mean_rmse,station_median_rmse,station_median_r,precision,recall,f1,n_test_stations
0,0,0.7895,0.7576,0.1589,0.7187,0.5858,0.8858,0.7298,0.6734,0.7005,8
1,1,0.8987,0.5840,0.1223,0.5724,0.5772,0.8974,0.8731,0.6910,0.7715,8
2,2,0.8662,0.6394,0.1340,0.6262,0.6154,0.8639,0.8496,0.6666,0.7471,8
3,3,0.8486,0.7134,0.1403,0.6933,0.6993,0.8594,0.8395,0.6756,0.7487,8
4,4,0.8662,0.6401,0.1300,0.6198,0.5750,0.8838,0.8421,0.6856,0.7558,8


── IDW mean over folds ──


,pearson_r,rmse,mae,station_mean_rmse,station_median_rmse,station_median_r,precision,recall,f1,n_test_stations
mean_over_folds,0.8538,0.6669,0.1371,0.6461,0.6105,0.8781,0.8268,0.6784,0.7447,8.0


In [2]:
# IDW: 

# ── IDW baseline on the Wagga gauge dataset ───────────────────────────────────
# Predicts each held-out test station from the rest of the gauge network via
# inverse-distance weighting. Uses the SAME folds (seed=123) and SAME metrics as
# the GNN runs, so the numbers drop straight into the gauge / +radar comparison.
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from sklearn.metrics import precision_recall_fscore_support

from src.raingauge.utils import load_raingauge_dataset
from src.sampling.main import stratified_spatial_kfold_dual

# --- knobs ---
POWER      = 2       # IDW distance exponent
N_NEAREST  = 10      # nearest source gauges to use (falls back to all available)
THRESHOLD  = 0.5     # mm/h wet/dry threshold for precision/recall/F1
FOLD_COUNT = 5

# --- 1. load Wagga gauge data (same paths as config_wagga_gauge.yaml) ---
gauge_df, meta_df = load_raingauge_dataset(
    rainfall_file="database/australia/wagga_rainfall_long.csv",
    metadata_file="database/australia/wagga_station_metadata.csv",
    start=2022, end=2022, uptime_threshold=0.9,
)
gauge_df.columns = gauge_df.columns.astype(str)                       # station_id cols
coords = {str(r.id): (r.latitude, r.longitude) for r in meta_df.itertuples()}

# --- 2. identical fold split ---
split_info = stratified_spatial_kfold_dual(
    meta_df, seed=123, plot=False, n_splits=FOLD_COUNT
)

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = np.radians(lat2 - lat1), np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def idw_predict(test_id, src_ids):
    """IDW time series for one test station from source gauges (NaN-aware)."""
    tlat, tlon = coords[test_id]
    d = np.array([haversine_km(tlat, tlon, *coords[s]) for s in src_ids])
    order = np.argsort(d)[:min(N_NEAREST, len(src_ids))]
    src = [src_ids[i] for i in order]
    w = 1.0 / np.maximum(d[order], 1e-6) ** POWER                     # (k,)
    S = gauge_df[src].to_numpy(dtype=float)                          # (T, k)
    mask = ~np.isnan(S)
    den = (mask * w[None, :]).sum(axis=1)
    num = (np.where(mask, S, 0.0) * w[None, :]).sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, num / den, np.nan)                  # (T,)

# --- 3. run IDW per fold, collect pooled + per-station predictions ---
per_fold_rows, all_p, all_t = [], [], []
for fold in range(FOLD_COUNT):
    src_ids  = [str(s) for s in split_info[fold]["statistical"]["train"]]   # all non-test gauges
    test_ids = [str(s) for s in split_info[fold]["ml"]["test"]]
    fp, ft, st_rmse, st_r = [], [], [], []
    for tid in test_ids:
        pred = idw_predict(tid, src_ids)
        targ = gauge_df[tid].to_numpy(dtype=float)
        m = (~np.isnan(pred)) & (~np.isnan(targ))
        if m.sum() < 2:
            continue
        p, t = pred[m], targ[m]
        fp.append(p); ft.append(t)
        st_rmse.append(np.sqrt(np.mean((p - t) ** 2)))
        st_r.append(pearsonr(p, t)[0] if p.std() > 0 and t.std() > 0 else np.nan)
    p = np.concatenate(fp); t = np.concatenate(ft)
    all_p.append(p); all_t.append(t)
    prec, rec, f1, _ = precision_recall_fscore_support(
        t >= THRESHOLD, p >= THRESHOLD, average="binary", zero_division=0)
    per_fold_rows.append({
        "fold": fold,
        "pearson_r":          pearsonr(p, t)[0],
        "rmse":               np.sqrt(np.mean((p - t) ** 2)),
        "mae":                np.mean(np.abs(p - t)),
        "station_mean_rmse":   float(np.mean(st_rmse)),
        "station_median_rmse": float(np.median(st_rmse)),
        "station_median_r":    float(np.nanmedian(st_r)),
        "precision": prec, "recall": rec, "f1": f1,
        "n_test_stations": len(st_rmse),
    })

idw_per_fold = pd.DataFrame(per_fold_rows)
idw_summary  = idw_per_fold.drop(columns=["fold"]).mean().to_frame("mean_over_folds").T
idw_per_fold.to_csv("idw_wagga_per_fold.csv", index=False)

print("── IDW per fold ──");  display(idw_per_fold.round(4))
print("── IDW mean over folds ──");  display(idw_summary.round(4))

Loading Australian raingauge data from database/australia/wagga_rainfall_long.csv
Pivoted dataframe shape (before uptime filter): (8760, 41)
Loading station metadata from database/australia/wagga_station_metadata.csv
Filtering by uptime threshold = 0.9
Dataframe shape after filter: (8760, 41)
Stations retained: 41
SPATIAL K-FOLD STRATIFIED SAMPLING (Fixed)
Total stations: 41
Clusters: 8
Folds: 5
Random seed: 123
--------------------------------------------------------------------------------
Cluster 0: 8 stations → 5 test assignments, 3 always-train
Cluster 1: 5 stations → 5 test assignments, 0 always-train
Cluster 2: 3 stations → 3 test assignments, 0 always-train
Cluster 3: 4 stations → 4 test assignments, 0 always-train
Cluster 4: 3 stations → 3 test assignments, 0 always-train
Cluster 5: 10 stations → 5 test assignments, 5 always-train
Cluster 6: 3 stations → 3 test assignments, 0 always-train
Cluster 7: 5 stations → 5 test assignments, 0 always-train

Creating Fold 1/5
Test statio

,fold,pearson_r,rmse,mae,station_mean_rmse,station_median_rmse,station_median_r,precision,recall,f1,n_test_stations
0,0,0.6477,0.6250,0.1139,0.6056,0.6257,0.6175,0.6290,0.7183,0.6707,8
1,1,0.6736,0.5755,0.1132,0.5656,0.5737,0.6772,0.6094,0.7421,0.6692,8
2,2,0.6126,0.6302,0.1127,0.6235,0.6220,0.6138,0.5933,0.6877,0.6370,8
3,3,0.7459,0.4587,0.0916,0.4422,0.3880,0.7267,0.6629,0.6675,0.6652,5
4,4,0.7710,0.4231,0.0811,0.4186,0.3984,0.7909,0.5976,0.8383,0.6977,4


── IDW mean over folds ──


,pearson_r,rmse,mae,station_mean_rmse,station_median_rmse,station_median_r,precision,recall,f1,n_test_stations
mean_over_folds,0.6901,0.5425,0.1025,0.5311,0.5216,0.6852,0.6185,0.7308,0.668,6.6


In [3]:
fold_count = 1
config_file = 'config.yaml'
with open(config_file) as f:
    config = yaml.safe_load(f)


FileNotFoundError: [Errno 2] No such file or directory: 'config.yaml'

In [ ]:
raingauge_df = load_raingauge_dataset(f'database/{config['dataset_parameters']['raingauge_file']}')

In [ ]:
raingauge_mappings = get_station_coordinate_mappings(start = 2021, end = 2025)

In [ ]:
filtered_stations = filter_uptime(raingauge_df, uptime_threshold = 0.9)
raingauge_df = raingauge_df[filtered_stations.keys()]
raingauge_df = raingauge_df.resample('15min').first() #resamples df to 15 mins
raingauge_mappings = {k:v for k, v in raingauge_mappings.items() if k in raingauge_df.keys()}

In [ ]:
#2. Get stratified training split
split_info = stratified_spatial_kfold_dual(
    raingauge_mappings, seed=123, plot=False, n_splits=fold_count
)


In [ ]:
print(raingauge_df.head(10))

In [ ]:
#3. Run the IDW for x folds
for fold in range(fold_count):
    training_gauges = split_info[fold]['statistical']['train']
    test_gauges = split_info[fold]['statistical']['test']

    # Run idw
    run_IDW_benchmark(raingauge_data=raingauge_df,
                      coordinates=raingauge_mappings,
                      training_stations=training_gauges,
                      test_stations=test_gauges,
                      power = 2,
                      n_nearest=15,
                      regression_plot=True
                      )

In [ ]:
#need to merge radar data with rain gauge data for KED

merged_df_KED = pd.merge(raingauge_choice_df, radar_df, on='time_sgt')
merged_df = raingauge_choice_df

# Kriging interpolation with rain gauge

In [ ]:
from matplotlib.colors import LogNorm

invalid_kriges = 0
training_ratio = config['dataset_parameters']['train_size']
station_names = []

for key in station_dict.keys():
  station_names.append(key,)

actual_values_arr = np.zeros(shape=[merged_df.shape[0], len(test_stations)])
predicted_values_arr = np.zeros(shape=[merged_df.shape[0], len(test_stations)])

start = time.time()

for idx in tqdm(range(len(merged_df_KED))):
  df = merged_df_KED.iloc[idx]

  kriging_result, kriging_variance = kriging_external_drift(df=df, 
                                                            station_names=training_stations, 
                                                            station_dict=station_dict, 
                                                            variogram_model='exponential', 
                                                            method='ordinary')

  if kriging_result is None:
    invalid_kriges += 1
    continue

    
  row_predicted_arr = []
  row_actual_arr = []
  
  #Calculate loss
  for test_station in test_stations:
    if not np.isnan(df[test_station]):
      rain_gauge_value = df[test_station]
      lat, lon = station_dict[test_station]
      row = math.floor((1.51 - lat) / 0.01)
      col = math.floor((lon - 103.6) / 0.01)
      kriged_value = kriging_result[row][col]

      row_actual_arr.append(rain_gauge_value)
      row_predicted_arr.append(max(0, kriged_value)) ## Kriged values can be negative and thus we need to account for this and set neg values to 0
    else:
      row_actual_arr.append(np.nan)
      row_predicted_arr.append(np.nan)
    
  actual_values_arr[idx] = np.array(row_actual_arr)
  predicted_values_arr[idx] = np.array(row_predicted_arr)



end = time.time()
MSE_arr = []

assert(len(actual_values_arr) == len(predicted_values_arr))

#calculate loss
for i in range(len(actual_values_arr)):
  pred = predicted_values_arr[i]
  act = actual_values_arr[i]
  mask = ~np.isnan(act)
  MSE = np.mean((pred[mask] - act[mask]) ** 2)
  MSE_arr.append(MSE)

average_RMSE_loss = np.sum(np.sqrt(np.array(MSE_arr))) / len(merged_df)
average_MSE_loss = np.sum(np.array(MSE_arr)) / len(merged_df)

print(f"invalid kriges: {invalid_kriges}")
print(f"final average loss: {average_RMSE_loss}")
#print(f"final average loss (0 rain = 0 loss): {total_RMSE_loss / (len(raingauge_choice_df))}")
print(f"Time taken = {end - start}")


plt.figure(figsize=(10,10))
actual = np.array(actual_values_arr).flatten()
predicted = np.array(predicted_values_arr).flatten()
mask = ~np.isnan(actual)

# plt.hist2d(x=actual[mask], y=predicted[mask],
#                   bins=100, 
#                   cmap='jet',
#                   cmin=1,
#                   norm=LogNorm(vmin=1, vmax=None))
plt.scatter(x=actual, y=predicted)

plot_bound = max(np.nanmax(actual).astype(int),np.nanmax(predicted).astype(int))
plt.plot(np.linspace(0,plot_bound,100),
        np.linspace(0,plot_bound,100))
plt.xlabel('actual_values')
plt.ylabel('predicted_values')
plt.show()


pearson_correlation, _ = pearsonr(actual[mask], predicted[mask])
print(f"Pearson correlation: {pearson_correlation}")
